## Text Classification using RNN & LSTM (Improved)

#### Name : Sufiyan Shaikh
####  Roll: SCFP125021

---

In [1]:
pip install -q tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

ModuleNotFoundError: No module named 'pandas'

In [340]:
df = pd.read_csv("IMDB Dataset.csv")
print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [341]:
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df[['review', 'sentiment', 'label']].head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [342]:
SAMPLE_SIZE = 16000
if SAMPLE_SIZE is not None:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# First split: 80% train_val, 20% test
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    df['review'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Second split: carve a validation set out of train_val
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=0.1,
    random_state=42,
    stratify=train_val_labels
)

print("Train:", len(train_texts), "Val:", len(val_texts), "Test:", len(test_texts))

Train: 11520 Val: 1280 Test: 3200


In [343]:
tokenizer = tiktoken.get_encoding("cl100k_base")
VOCAB_SIZE = tokenizer.n_vocab
PAD_IDX = VOCAB_SIZE
MAX_LEN = 200

print("Vocab size:", VOCAB_SIZE)
print("Pad index:", PAD_IDX)

Vocab size: 100277
Pad index: 100277


In [344]:
def encode_and_pad(text, max_len=MAX_LEN):
    ids = tokenizer.encode(text, disallowed_special=())
    true_len = max(min(len(ids), max_len), 1)
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids, true_len

sample_ids, sample_len = encode_and_pad(train_texts[0])
print(sample_len, sample_ids[:20])

200 [43, 290, 25623, 706, 3686, 311, 12391, 5196, 13, 7357, 3254, 5818, 505, 69132, 25623, 706, 1027, 671, 1065, 14991]


In [345]:
train_encoded = [encode_and_pad(t) for t in train_texts]
val_encoded = [encode_and_pad(t) for t in val_texts]
test_encoded = [encode_and_pad(t) for t in test_texts]

train_ids = [e[0] for e in train_encoded]
train_lengths = [e[1] for e in train_encoded]
val_ids = [e[0] for e in val_encoded]
val_lengths = [e[1] for e in val_encoded]
test_ids = [e[0] for e in test_encoded]
test_lengths = [e[1] for e in test_encoded]

print(len(train_ids), len(val_ids), len(test_ids))

11520 1280 3200


In [346]:
class IMDBDataset(Dataset):
    def __init__(self, ids_list, lengths, labels):
        self.X = torch.tensor(ids_list, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx], self.y[idx]

train_dataset = IMDBDataset(train_ids, train_lengths, train_labels)
val_dataset = IMDBDataset(val_ids, val_lengths, val_labels)
test_dataset = IMDBDataset(test_ids, test_lengths, test_labels)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

xb, lb, yb = next(iter(train_loader))
print(xb.shape, lb.shape, yb.shape)

torch.Size([32, 200]) torch.Size([32]) torch.Size([32])


-----------------

### Activity 1 : Text Classification using RNN

In [347]:
from torch.nn.utils.rnn import pack_padded_sequence

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, pad_idx=PAD_IDX, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        out = self.dropout(hidden.squeeze(0))
        out = self.fc(out)
        return out.squeeze(1)

rnn_model = RNNClassifier(VOCAB_SIZE).to(device)
print(rnn_model)

RNNClassifier(
  (embedding): Embedding(100278, 64, padding_idx=100277)
  (rnn): RNN(64, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [348]:
def train_model(model, train_loader, val_loader, epochs=8, lr=1e-3, clip=5.0, weight_decay=1e-5):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for xb, lb, yb in train_loader:
            xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb, lb)
            loss = criterion(preds, yb)
            loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
        avg_loss = total_loss / len(train_loader.dataset)

        # validation accuracy for this epoch
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, lb, yb in val_loader:
                xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
                preds = model(xb, lb)
                predicted = (torch.sigmoid(preds) >= 0.5).float()
                correct += (predicted == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_loss:.4f} - Val Accuracy: {val_acc:.4f}")

    return model

In [349]:
rnn_model = train_model(rnn_model, train_loader, val_loader, epochs=8, lr=1e-3)

Epoch 1/8 - Train Loss: 0.6954 - Val Accuracy: 0.5219
Epoch 2/8 - Train Loss: 0.6773 - Val Accuracy: 0.5813
Epoch 3/8 - Train Loss: 0.6485 - Val Accuracy: 0.5953
Epoch 4/8 - Train Loss: 0.6428 - Val Accuracy: 0.6203
Epoch 5/8 - Train Loss: 0.5646 - Val Accuracy: 0.6508
Epoch 6/8 - Train Loss: 0.4838 - Val Accuracy: 0.6070
Epoch 7/8 - Train Loss: 0.4122 - Val Accuracy: 0.6813
Epoch 8/8 - Train Loss: 0.3293 - Val Accuracy: 0.6922


In [350]:
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, lb, yb in test_loader:
            xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
            preds = model(xb, lb)
            predicted_labels = (torch.sigmoid(preds) >= 0.5).float()
            correct += (predicted_labels == yb).sum().item()
            total += yb.size(0)
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")
    return accuracy

rnn_accuracy = evaluate_model(rnn_model, test_loader)

Test Accuracy: 0.6634


In [351]:
def predict_sentiment(model, sentence):
    model.eval()
    ids, length = encode_and_pad(sentence)
    x = torch.tensor([ids], dtype=torch.long).to(device)
    l = torch.tensor([length], dtype=torch.long).to(device)
    with torch.no_grad():
        pred = torch.sigmoid(model(x, l)).item()
    label = "positive" if pred >= 0.5 else "negative"
    return label, pred

print("=== RNN Predictions ===")
for s in test_sentences:
    label, score = predict_sentiment(rnn_model, s)
    print(f"'{s}' -> {label} (score={score:.3f})")

=== RNN Predictions ===
'This movie was an absolute masterpiece, I loved every second of it.' -> negative (score=0.476)
'Terrible acting and a boring plot, I want my two hours back.' -> negative (score=0.031)
'It was okay, not great but not bad either.' -> negative (score=0.031)


------------------

### Activity 2 : Text Classification using LSTM

In [352]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, pad_idx=PAD_IDX, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, cell) = self.lstm(packed)
        out = self.dropout(hidden.squeeze(0))
        out = self.fc(out)
        return out.squeeze(1)

lstm_model = LSTMClassifier(VOCAB_SIZE).to(device)
print(lstm_model)

LSTMClassifier(
  (embedding): Embedding(100278, 64, padding_idx=100277)
  (lstm): LSTM(64, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [354]:
lstm_model = train_model(lstm_model, train_loader, val_loader, epochs=8, lr=1e-3)

Epoch 1/8 - Train Loss: 0.6827 - Val Accuracy: 0.6016
Epoch 2/8 - Train Loss: 0.6570 - Val Accuracy: 0.5906
Epoch 3/8 - Train Loss: 0.5965 - Val Accuracy: 0.6367
Epoch 4/8 - Train Loss: 0.4989 - Val Accuracy: 0.7547
Epoch 5/8 - Train Loss: 0.3540 - Val Accuracy: 0.7852
Epoch 6/8 - Train Loss: 0.2300 - Val Accuracy: 0.8195
Epoch 7/8 - Train Loss: 0.1393 - Val Accuracy: 0.8289
Epoch 8/8 - Train Loss: 0.0813 - Val Accuracy: 0.8258


In [355]:
lstm_accuracy = evaluate_model(lstm_model, test_loader)

Test Accuracy: 0.8081


In [356]:
print("=== LSTM Predictions ===")
for s in test_sentences:
    label, score = predict_sentiment(lstm_model, s)
    print(f"'{s}' -> {label} (score={score:.3f})")

print("\n=== Comparison ===")
print(f"RNN Test Accuracy:  {rnn_accuracy:.4f}")
print(f"LSTM Test Accuracy: {lstm_accuracy:.4f}")

=== LSTM Predictions ===
'This movie was an absolute masterpiece, I loved every second of it.' -> positive (score=0.997)
'Terrible acting and a boring plot, I want my two hours back.' -> negative (score=0.003)
'It was okay, not great but not bad either.' -> negative (score=0.028)

=== Comparison ===
RNN Test Accuracy:  0.6634
LSTM Test Accuracy: 0.8081


--------------------